# NBA Game-Prediction Dataset Builder (NBA API, last 10 seasons)

Replaces the SQLite-from-Kaggle source with **live data from `nba_api`**, so it stays current.

**Two phases:**

1. **Phase 1 — Box scores.** Fetch team-level game logs from the NBA stats API for each season, stack them, and produce `box_data.csv` in the same T1/T2 layout (T1=home, T2=away).
2. **Phase 2 — Pre-game features.** Identical leakage-free pipeline as the previous notebook: each game's features use only that team's prior games in the same season.

**Outputs:**
- `box_data.csv` — game-level T1/T2 box scores
- `matchups.csv` — pre-game season-to-date features for both teams + `Target_Win`

**Phase 1 caches each season's raw fetch to `raw_seasons/`** so you don't pay the network cost twice if you re-run.


## Setup

In [1]:
# One-time install if you don't have it:
# !pip install nba_api pandas numpy

import time
from pathlib import Path

import numpy as np
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog

# ---- Config ----
# 10 seasons (last full decade). Edit if you want a different range.
SEASONS = [f"{y}-{str(y+1)[2:]}" for y in range(2016, 2026)]   # 2016-17 .. 2025-26
SEASON_TYPE = "Regular Season"   # or "Playoffs"
RAW_DIR     = Path("raw_seasons")
OUT_DIR     = Path(".")
RAW_DIR.mkdir(exist_ok=True)

SLEEP_BETWEEN_CALLS = 0.6        # be polite to stats.nba.com
RETRIES             = 4

# Same Elo constants as before
ELO_BASE      = 1500.0
ELO_K         = 20.0
ELO_HOME_ADV  = 100.0
ELO_REGRESS   = 0.25

STAT_COLS = ['Score','FGM','FGA','FGM3','FGA3','FTM','FTA',
             'OR','DR','Ast','TO','Stl','Blk','PF']
OPP_COLS  = [f'Opp_{s}' for s in STAT_COLS]

print("Seasons to fetch:", SEASONS)

Seasons to fetch: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


## Phase 1 — Fetch & build `box_data.csv`

`LeagueGameLog` returns one row per (team, game), so 2 rows per game. We pivot home vs. away (the `MATCHUP` field uses `vs.` for home and `@` for away) and merge into one row per game with T1_*/T2_* columns.

### Helper: fetch one season with retries + cache

In [2]:
def fetch_season(season: str,
                 season_type: str = SEASON_TYPE,
                 retries: int = RETRIES,
                 cache_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Pull team game logs for a single season. Caches to a CSV in cache_dir."""
    fname = cache_dir / f"{season}_{season_type.replace(' ', '_')}.csv"
    if fname.exists():
        return pd.read_csv(fname)

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season,
                season_type_all_star=season_type,
                player_or_team_abbreviation='T',  # team-level
                timeout=60,
            )
            df = log.get_data_frames()[0]
            df.to_csv(fname, index=False)
            return df
        except Exception as e:
            last_err = e
            wait = 5 * attempt
            print(f"  [{season}] attempt {attempt}/{retries} failed: {e}; "
                  f"retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {season} after {retries} attempts: {last_err}")

### Run the fetch loop

In [3]:
raw_frames = []
for s in SEASONS:
    print(f"Fetching {s} ...", end=" ", flush=True)
    df = fetch_season(s)
    print(f"{len(df):,} team-game rows")
    raw_frames.append(df)
    time.sleep(SLEEP_BETWEEN_CALLS)

raw = pd.concat(raw_frames, ignore_index=True)
print(f"\nTotal team-game rows: {len(raw):,}")
raw.head()

Fetching 2016-17 ... 2,460 team-game rows
Fetching 2017-18 ... 2,460 team-game rows
Fetching 2018-19 ... 2,460 team-game rows
Fetching 2019-20 ... 2,118 team-game rows
Fetching 2020-21 ... 2,160 team-game rows
Fetching 2021-22 ... 2,460 team-game rows
Fetching 2022-23 ... 2,460 team-game rows
Fetching 2023-24 ... 2,460 team-game rows
Fetching 2024-25 ... 2,460 team-game rows
Fetching 2025-26 ... 2,460 team-game rows

Total team-game rows: 23,958


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22016,1610612752,NYK,New York Knicks,21600001,2016-10-25,NYK @ CLE,L,240,32,...,29,42,17,6,6,18,22,88,-29,1
1,22016,1610612757,POR,Portland Trail Blazers,21600002,2016-10-25,POR vs. UTA,W,240,39,...,29,34,22,5,3,13,18,113,9,1
2,22016,1610612759,SAS,San Antonio Spurs,21600003,2016-10-25,SAS @ GSW,W,240,47,...,34,55,25,13,3,14,19,129,29,1
3,22016,1610612739,CLE,Cleveland Cavaliers,21600001,2016-10-25,CLE vs. NYK,W,240,45,...,40,51,31,12,5,15,22,117,29,1
4,22016,1610612744,GSW,Golden State Warriors,21600003,2016-10-25,GSW vs. SAS,L,240,40,...,27,35,24,11,6,16,19,100,-29,1


### Helper: stack team-game rows into the T1/T2 box layout

`MATCHUP` like `"LAL vs. BOS"` = home (T1), `"LAL @ BOS"` = away (T2). We split, rename columns, and inner-join on `GAME_ID`.

In [4]:
# Map of nba_api column names -> our schema (without the T1_/T2_ prefix)
NBA_TO_OURS = {
    'TEAM_ID':   'TeamID',
    'TEAM_NAME': 'TeamName',
    'PTS':  'Score',
    'FGM':  'FGM',  'FGA':  'FGA',
    'FG3M': 'FGM3', 'FG3A': 'FGA3',
    'FTM':  'FTM',  'FTA':  'FTA',
    'OREB': 'OR',   'DREB': 'DR',
    'AST':  'Ast',  'TOV':  'TO',
    'STL':  'Stl',  'BLK':  'Blk',
    'PF':   'PF',
}

def build_box_data(raw: pd.DataFrame) -> pd.DataFrame:
    """Convert nba_api LeagueGameLog rows into the T1/T2 box layout."""
    df = raw.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    # SEASON_ID like '22015' -> Season = 2015
    df['Season'] = df['SEASON_ID'].astype(str).str[-4:].astype(int)
    
    # 1. Use "@" to reliably flag away teams
    df['IS_AWAY'] = df['MATCHUP'].str.contains('@', regex=False)
    
    # 2. Lenient filter for valid matchups
    df = df[df['MATCHUP'].str.contains('vs|@', regex=True, case=False)].copy()
    df = df.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])

    # 3. Drop rows missing core box-score numbers BEFORE pairing them up
    needed = list(NBA_TO_OURS.keys()) + ['WL']
    df = df.dropna(subset=needed)

    # 4. Force exactly 1 home and 1 away per GAME_ID to prevent MergeError
    # Sorting by IS_AWAY puts the False (home) row first.
    df = df.sort_values(['GAME_ID', 'IS_AWAY'])
    df['row_num'] = df.groupby('GAME_ID').cumcount()
    
    # Keep only games that successfully retained exactly 2 teams
    valid_games = df['GAME_ID'].value_counts()
    valid_games = valid_games[valid_games == 2].index
    df = df[df['GAME_ID'].isin(valid_games)].copy()

    df['IS_HOME'] = df['row_num'] == 0

    cols_to_keep = ['GAME_ID', 'GAME_DATE', 'Season', 'WL'] + list(NBA_TO_OURS.keys())

    home = (df[df['IS_HOME']][cols_to_keep]
            .rename(columns={**{k: f'T1_{v}' for k, v in NBA_TO_OURS.items()},
                             'WL': 'T1_WL'}))
    away = (df[~df['IS_HOME']][['GAME_ID', 'WL'] + list(NBA_TO_OURS.keys())]
            .rename(columns={**{k: f'T2_{v}' for k, v in NBA_TO_OURS.items()},
                             'WL': 'T2_WL'}))

    # The one-to-one merge will now flawlessly succeed
    box = home.merge(away, on='GAME_ID', validate='one_to_one')

    box = box.rename(columns={'GAME_ID': 'GameID', 'GAME_DATE': 'DayDate'})
    box['T1_Loc']     = 'H'
    box['Target_Win'] = (box['T1_WL'] == 'W').astype(int)
    box = box.drop(columns=['T1_WL', 'T2_WL'])

    # Order columns
    front = ['Season', 'DayDate', 'GameID',
             'T1_TeamID', 'T1_TeamName', 'T2_TeamID', 'T2_TeamName', 'T1_Loc']
    t1_stats = [f'T1_{c}' for c in STAT_COLS]
    t2_stats = [f'T2_{c}' for c in STAT_COLS]
    box = box[front + t1_stats + t2_stats + ['Target_Win']]

    box = box.sort_values(['DayDate', 'GameID']).reset_index(drop=True)
    return box

In [5]:
box = build_box_data(raw)
box.to_csv("box_data.csv", index=False)
print(f"Wrote box_data.csv ({len(box):,} games, "
      f"seasons {box['Season'].min()}-{box['Season'].max()})")
box.head()

Wrote box_data.csv (11,979 games, seasons 2016-2025)


,Season,DayDate,GameID,T1_TeamID,T1_TeamName,T2_TeamID,T2_TeamName,T1_Loc,T1_Score,T1_FGM,...,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,Target_Win
0,2016,2016-10-25,21600001,1610612739,Cleveland Cavaliers,1610612752,New York Knicks,H,117,45,...,15,20,13,29,17,18,6,6,22,1
1,2016,2016-10-25,21600002,1610612757,Portland Trail Blazers,1610612762,Utah Jazz,H,113,39,...,16,16,6,25,19,14,9,5,19,1
2,2016,2016-10-25,21600003,1610612744,Golden State Warriors,1610612759,San Antonio Spurs,H,100,40,...,23,26,21,34,25,14,13,3,19,0
3,2016,2016-10-26,21600004,1610612753,Orlando Magic,1610612748,Miami Heat,H,96,34,...,10,16,16,36,27,12,5,7,22,0
4,2016,2016-10-26,21600005,1610612754,Indiana Pacers,1610612742,Dallas Mavericks,H,130,47,...,13,18,10,39,26,15,8,8,27,1


## Phase 2 — Pre-game features

Identical leakage-free pipeline from before. For every `(GameID, TeamID)`, compute season-to-date averages, win pcts, advanced metrics, last-14-day form, pre-game Elo, and a snapshot seed — all using **only games strictly before** the current game in the same season.

The first game of each team-season has no priors, so its features come out as `NaN`.

### Helpers

In [6]:
def explode_to_team_perspective(box: pd.DataFrame) -> pd.DataFrame:
    """Each game becomes 2 rows -- one from each team's perspective."""
    def side(prefix_self, prefix_opp, loc):
        d = pd.DataFrame({
            'Season' : box['Season'],
            'DayDate': box['DayDate'],
            'GameID' : box['GameID'],
            'TeamID' : box[f'{prefix_self}_TeamID'],
            'OppID'  : box[f'{prefix_opp}_TeamID'],
            'Loc'    : loc,
            'Won'    : box['Target_Win'] if prefix_self == 'T1' else 1 - box['Target_Win'],
        })
        for s in STAT_COLS:
            d[s]          = box[f'{prefix_self}_{s}']
            d[f'Opp_{s}'] = box[f'{prefix_opp}_{s}']
        return d

    return pd.concat([side('T1', 'T2', 'H'),
                      side('T2', 'T1', 'A')], ignore_index=True)


def compute_possessions(g: pd.DataFrame) -> pd.DataFrame:
    """Per-game possessions (Poss = FGA - OR + TO + 0.475*FTA)."""
    g = g.copy()
    g['Poss']    = g['FGA']     - g['OR']     + g['TO']     + 0.475 * g['FTA']
    g['OppPoss'] = g['Opp_FGA'] - g['Opp_OR'] + g['Opp_TO'] + 0.475 * g['Opp_FTA']
    return g

In [7]:
def compute_elo_pregame(box: pd.DataFrame,
                        base: float = ELO_BASE,
                        k: float = ELO_K,
                        hca: float = ELO_HOME_ADV,
                        regress: float = ELO_REGRESS) -> pd.DataFrame:
    """One row per game with both teams' pre-game Elo."""
    elo: dict = {}
    records = []
    prev_season = None

    for _, row in box[['Season','GameID','T1_TeamID','T2_TeamID','Target_Win']].iterrows():
        season = row['Season']
        if prev_season is not None and season != prev_season:
            for tid in elo:
                elo[tid] = (1 - regress) * elo[tid] + regress * base
        prev_season = season

        t1, t2 = row['T1_TeamID'], row['T2_TeamID']
        e1, e2 = elo.get(t1, base), elo.get(t2, base)

        # snapshot BEFORE the game updates Elo
        records.append({
            'GameID'        : row['GameID'],
            'T1_TeamID'     : t1,
            'T2_TeamID'     : t2,
            'T1_Pregame_Elo': e1,
            'T2_Pregame_Elo': e2,
        })

        exp1   = 1.0 / (1.0 + 10 ** (-((e1 + hca) - e2) / 400.0))
        delta  = k * (row['Target_Win'] - exp1)
        elo[t1] = e1 + delta
        elo[t2] = e2 - delta

    return pd.DataFrame(records)

### Main computation

In [8]:
def compute_pregame_features(box: pd.DataFrame) -> pd.DataFrame:
    tg = explode_to_team_perspective(box)
    tg = compute_possessions(tg)
    tg = tg.sort_values(['TeamID', 'Season', 'DayDate', 'GameID']).reset_index(drop=True)

    grp_keys = ['TeamID', 'Season']
    grp = tg.groupby(grp_keys)

    # ---- Games played BEFORE this one (0 for first game of season) ----
    tg['Games_Played'] = grp.cumcount()

    # ---- Cumulative totals BEFORE this game (cumsum - current) ----
    sum_cols = STAT_COLS + OPP_COLS + ['Poss', 'OppPoss']
    for c in sum_cols:
        tg[f'cum_{c}'] = grp[c].cumsum() - tg[c]
    tg['cum_Wins'] = grp['Won'].cumsum() - tg['Won']

    # ---- Per-game averages ----
    safe_games = tg['Games_Played'].replace(0, np.nan)
    for c in STAT_COLS + OPP_COLS:
        tg[f'Avg_{c}'] = tg[f'cum_{c}'] / safe_games

    # ---- Advanced metrics on cumulative totals ----
    safe_poss     = tg['cum_Poss'].replace(0, np.nan)
    safe_opp_poss = tg['cum_OppPoss'].replace(0, np.nan)
    safe_fga      = tg['cum_FGA'].replace(0, np.nan)
    safe_or_dr    = (tg['cum_OR'] + tg['cum_Opp_DR']).replace(0, np.nan)

    tg['Avg_Off_Eff'] = (tg['cum_Score']     / safe_poss)     * 100   # OffEff
    tg['Avg_Def_Eff'] = (tg['cum_Opp_Score'] / safe_opp_poss) * 100   # DefEff
    tg['Net_Rating']  = tg['Avg_Off_Eff'] - tg['Avg_Def_Eff']         # NetEff
    tg['EFG_Pct']     = (tg['cum_FGM'] + 0.5 * tg['cum_FGM3']) / safe_fga
    tg['TO_Rate']     = tg['cum_TO'] / safe_poss
    tg['OR_Pct']      = tg['cum_OR'] / safe_or_dr

    # ---- Pythagorean Win Percentage ----
    # Formula: (Points For ^ 14) / (Points For ^ 14 + Points Against ^ 14)
    exponent = 14
    pf_14 = tg['Avg_Score'] ** exponent
    pa_14 = tg['Avg_Opp_Score'] ** exponent
    # Fill with 0.5 (50% expected win rate) for the first game of the season to prevent NaNs
    tg['Pythag_Win_Pct'] = (pf_14 / (pf_14 + pa_14)).fillna(0.5)

    # ---- Win pct (overall / home / away) BEFORE current game (with Laplace Smoothing) ----
    alpha = 1.0  # Tune this! Higher alpha = stronger pull to 50%
    
    # We can use Games_Played directly now instead of safe_games because 
    # the denominator will be (0 + 2*alpha), preventing division by zero.
    tg['Win_Pct']         = (tg['cum_Wins'] + alpha) / (tg['Games_Played'] + 2 * alpha)
    tg['Overall_Win_Pct'] = tg['Win_Pct']

    tg['_is_home']  = (tg['Loc'] == 'H').astype(int)
    tg['_is_away']  = (tg['Loc'] == 'A').astype(int)
    tg['_home_won'] = tg['Won'] * tg['_is_home']
    tg['_away_won'] = tg['Won'] * tg['_is_away']
    g2 = tg.groupby(grp_keys)
    cum_hg = g2['_is_home'].cumsum()  - tg['_is_home']
    cum_hw = g2['_home_won'].cumsum() - tg['_home_won']
    cum_ag = g2['_is_away'].cumsum()  - tg['_is_away']
    cum_aw = g2['_away_won'].cumsum() - tg['_away_won']
    
    tg['Home_Win_Pct'] = (cum_hw + alpha) / (cum_hg + 2 * alpha)
    tg['Away_Win_Pct'] = (cum_aw + alpha) / (cum_ag + 2 * alpha)
    tg = tg.drop(columns=['_is_home', '_is_away', '_home_won', '_away_won'])

    # ---- Last-14-day win pct (rolling time window, current row excluded) ----
    last14 = np.full(len(tg), np.nan)
    for _, idx in tg.groupby(grp_keys, sort=False).indices.items():
        sub = tg.iloc[idx]
        s = pd.Series(sub['Won'].values,
                      index=pd.DatetimeIndex(sub['DayDate']))
        last14[idx] = s.rolling('14D', closed='left').mean().values
    tg['Last_14_Days_Win_Pct'] = last14

    # ---- Pre-game Elo (merged from chronological walk) ----
    elo_df = compute_elo_pregame(box)
    elo_t1 = elo_df[['GameID','T1_TeamID','T1_Pregame_Elo']].rename(
        columns={'T1_TeamID': 'TeamID', 'T1_Pregame_Elo': 'Pregame_Elo'})
    elo_t2 = elo_df[['GameID','T2_TeamID','T2_Pregame_Elo']].rename(
        columns={'T2_TeamID': 'TeamID', 'T2_Pregame_Elo': 'Pregame_Elo'})
    elo_long = pd.concat([elo_t1, elo_t2], ignore_index=True)
    tg = tg.merge(elo_long, on=['GameID', 'TeamID'], how='left')

    # ---- Seed: rank by current Win_Pct within season as of game date ----
    seed_parts = []
    for season, sub in tg.groupby('Season'):
        pvt = (sub.pivot_table(index='DayDate', columns='TeamID',
                               values='Win_Pct', aggfunc='mean')
                  .sort_index().ffill())
        ranks = pvt.rank(axis=1, method='min', ascending=False)
        long = (ranks.reset_index()
                     .melt(id_vars='DayDate', var_name='TeamID', value_name='seed'))
        long['Season'] = season
        seed_parts.append(long)
    seeds = pd.concat(seed_parts, ignore_index=True)
    tg = tg.merge(seeds, on=['Season','TeamID','DayDate'], how='left')

    # ---> Note: 'Pythag_Win_Pct' added to the keep list below! <---
    keep = (['Season','DayDate','GameID','TeamID','OppID','Loc','Won','Games_Played']
            + [f'Avg_{c}' for c in STAT_COLS + OPP_COLS]
            + ['Avg_Off_Eff','Avg_Def_Eff','Net_Rating','EFG_Pct','TO_Rate','OR_Pct',
               'Win_Pct','Overall_Win_Pct','Home_Win_Pct','Away_Win_Pct',
               'Last_14_Days_Win_Pct','Pregame_Elo','seed', 'Pythag_Win_Pct']) 
    return tg[keep].reset_index(drop=True)

In [9]:
pregame = compute_pregame_features(box)
print(f"Pre-game features: {len(pregame):,} rows ({len(pregame)//2:,} games x 2 teams)")
print(f"Rows with NaN features (first game of each team-season): "
      f"{pregame['Avg_Score'].isna().sum():,}")
pregame

Pre-game features: 23,958 rows (11,979 games x 2 teams)
Rows with NaN features (first game of each team-season): 300


,Season,DayDate,GameID,TeamID,OppID,Loc,Won,Games_Played,Avg_Score,Avg_FGM,...,TO_Rate,OR_Pct,Win_Pct,Overall_Win_Pct,Home_Win_Pct,Away_Win_Pct,Last_14_Days_Win_Pct,Pregame_Elo,seed,Pythag_Win_Pct
0,2016,2016-10-27,21600014,1610612737,1610612764,H,1,0,NaN,NaN,...,NaN,NaN,0.500000,0.500000,0.500000,0.500000,NaN,1500.000000,5.0,0.500000
1,2016,2016-10-29,21600026,1610612737,1610612755,A,1,1,114.000000,44.000000,...,0.202801,0.333333,0.666667,0.666667,0.666667,0.500000,1.000000,1507.198700,4.0,0.878158
2,2016,2016-10-31,21600044,1610612737,1610612758,H,1,2,109.000000,43.000000,...,0.160603,0.240964,0.750000,0.750000,0.666667,0.666667,1.000000,1519.461436,2.0,0.967693
3,2016,2016-11-02,21600059,1610612737,1610612747,H,0,3,108.000000,39.666667,...,0.151915,0.280303,0.800000,0.800000,0.750000,0.666667,1.000000,1526.333271,2.0,0.940557
4,2016,2016-11-04,21600070,1610612737,1610612764,A,0,4,110.000000,40.000000,...,0.156260,0.271676,0.666667,0.666667,0.600000,0.666667,0.750000,1512.493553,5.0,0.848738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23953,2025,2026-04-03,22501120,1610612766,1610612754,H,1,77,116.233766,40.974026,...,0.151633,0.304916,0.531646,0.531646,0.512195,0.550000,0.714286,1542.299413,15.0,0.645855
23954,2025,2026-04-05,22501137,1610612766,1610612750,A,1,78,116.397436,41.038462,...,0.150963,0.304294,0.537500,0.537500,0.523810,0.550000,0.714286,1545.269897,15.0,0.651575
23955,2025,2026-04-07,22501149,1610612766,1610612738,A,0,79,116.468354,41.088608,...,0.150986,0.305270,0.543210,0.543210,0.523810,0.560976,0.750000,1558.801832,15.0,0.654700
23956,2025,2026-04-10,22501171,1610612766,1610612765,H,0,80,116.287500,41.050000,...,0.150855,0.305257,0.536585,0.536585,0.523810,0.547619,0.571429,1554.354598,15.0,0.649144


## Phase 3 — Build `matchups.csv`

Merge each game with both teams' pre-game features, in the column order from your spec (with `End_Season_Elo` -> `Pregame_Elo`, since the Elo is now snapshotted before each game rather than at season-end).

In [10]:
FEATURE_COLS = [
    'Avg_Off_Eff', 'Avg_Def_Eff', 'Net_Rating',
    'Win_Pct', 'Pregame_Elo',
    'Overall_Win_Pct', 'Home_Win_Pct', 'Away_Win_Pct',
    'Avg_Score', 'Avg_FGM', 'Avg_FGA', 'Avg_FGM3', 'Avg_FGA3',
    'Avg_FTM',   'Avg_FTA', 'Avg_OR',  'Avg_DR',
    'Avg_Ast',   'Avg_TO',  'Avg_Stl', 'Avg_Blk', 'Avg_PF',
    'Avg_Opp_Score', 'Avg_Opp_FGM', 'Avg_Opp_FGA',
    'Avg_Opp_FGM3',  'Avg_Opp_FGA3',
    'Avg_Opp_FTM',   'Avg_Opp_FTA',
    'Avg_Opp_OR',    'Avg_Opp_DR',   'Avg_Opp_Ast',
    'Avg_Opp_TO',    'Avg_Opp_Stl',  'Avg_Opp_Blk', 'Avg_Opp_PF',
    'seed', 'Last_14_Days_Win_Pct','Pythag_Win_Pct'
]


def build_symmetrical_matchups(box: pd.DataFrame, pregame: pd.DataFrame) -> pd.DataFrame:
    # 1. Start with the exploded team logs (2 rows per game, 24,000+ total rows)
    base = explode_to_team_perspective(box)[['Season', 'DayDate', 'GameID', 'TeamID', 'OppID', 'Loc', 'Won']]
    
    # Grab just the features we want to merge
    feats = pregame[['GameID', 'TeamID'] + FEATURE_COLS]
    
    # 2. Merge the Focal Team's pregame features
    m = base.merge(feats, on=['GameID', 'TeamID'], how='inner')
    
    # 3. Merge the Opponent's pregame features
    # Rename columns so they attach properly to the OppID
    opp_feats = feats.rename(columns={
        'TeamID': 'OppID', 
        **{c: f'Opp_{c}' for c in FEATURE_COLS}
    })
    m = m.merge(opp_feats, on=['GameID', 'OppID'], how='inner')
    
    # 4. Calculate Difference Features (Team - Opponent)
    diff_cols = []
    for c in FEATURE_COLS:
        diff_name = f'Diff_{c}'
        m[diff_name] = m[c] - m[f'Opp_{c}']
        diff_cols.append(diff_name)
        
    # 5. Add our Home Court indicator and rename the Target
    m['Is_Home'] = (m['Loc'] == 'H').astype(int)
    m = m.rename(columns={'Won': 'Target_Win'})
    
    # 6. Keep only metadata, the Home indicator, the differences, and the target
    final_cols = ['Season', 'DayDate', 'GameID', 'TeamID', 'OppID', 'Is_Home'] + diff_cols + ['Target_Win']
    m = m[final_cols]
    
    # Sort chronologically
    return m.sort_values(['DayDate', 'GameID']).reset_index(drop=True)

# Build and save the new dataset
matchups_sym = build_symmetrical_matchups(box, pregame)

# Drop first games of the season (NaNs)
matchups_sym = matchups_sym.dropna().copy()

matchups_sym.to_csv("matchups.csv", index=False)
print(f"Wrote matchups.csv ({len(matchups_sym):,} rows, {len(matchups_sym.columns)} cols)")
matchups_sym.head()

Wrote matchups.csv (23,616 rows, 46 cols)


,Season,DayDate,GameID,TeamID,OppID,Is_Home,Diff_Avg_Off_Eff,Diff_Avg_Def_Eff,Diff_Net_Rating,Diff_Win_Pct,...,Diff_Avg_Opp_DR,Diff_Avg_Opp_Ast,Diff_Avg_Opp_TO,Diff_Avg_Opp_Stl,Diff_Avg_Opp_Blk,Diff_Avg_Opp_PF,Diff_seed,Diff_Last_14_Days_Win_Pct,Diff_Pythag_Win_Pct,Target_Win
30,2016,2016-10-27,21600016,1610612758,1610612759,1,-14.413106,-2.383702,-12.029404,0.000000,...,4.0,-4.0,2.0,-5.0,-3.0,10.0,0.0,0.0,-0.043095,0
31,2016,2016-10-27,21600016,1610612759,1610612758,0,14.413106,2.383702,12.029404,0.000000,...,-4.0,4.0,-2.0,5.0,3.0,-10.0,0.0,0.0,0.043095,1
34,2016,2016-10-28,21600018,1610612761,1610612739,1,2.160188,3.892827,-1.732640,0.000000,...,3.0,0.0,-4.0,2.0,-6.0,2.0,0.0,0.0,-0.055794,0
35,2016,2016-10-28,21600018,1610612739,1610612761,0,-2.160188,-3.892827,1.732640,0.000000,...,-3.0,0.0,4.0,-2.0,6.0,-2.0,0.0,0.0,0.055794,1
36,2016,2016-10-28,21600019,1610612751,1610612754,1,-2.131634,13.755657,-15.887291,-0.333333,...,-4.0,10.0,4.0,5.0,1.0,-7.0,21.0,-1.0,-0.374340,1


In [11]:
matchups = build_symmetrical_matchups(box, pregame)
matchups.to_csv(OUT_DIR / "matchups.csv", index=False)
print(f"Wrote matchups.csv ({len(matchups):,} rows, {len(matchups.columns)} cols)")

# Optional: drop the first-game-of-season rows that have NaN features
clean = matchups.dropna(subset=[c for c in matchups.columns
                                if c.startswith(('T1_Avg', 'T2_Avg'))])
print(f"Rows after dropping NaN-feature games: {len(clean):,}")
matchups

Wrote matchups.csv (23,958 rows, 46 cols)
Rows after dropping NaN-feature games: 23,958


,Season,DayDate,GameID,TeamID,OppID,Is_Home,Diff_Avg_Off_Eff,Diff_Avg_Def_Eff,Diff_Net_Rating,Diff_Win_Pct,...,Diff_Avg_Opp_DR,Diff_Avg_Opp_Ast,Diff_Avg_Opp_TO,Diff_Avg_Opp_Stl,Diff_Avg_Opp_Blk,Diff_Avg_Opp_PF,Diff_seed,Diff_Last_14_Days_Win_Pct,Diff_Pythag_Win_Pct,Target_Win
0,2016,2016-10-25,21600001,1610612739,1610612752,1,NaN,NaN,NaN,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.000000,1
1,2016,2016-10-25,21600001,1610612752,1610612739,0,NaN,NaN,NaN,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.000000,0
2,2016,2016-10-25,21600002,1610612757,1610612762,1,NaN,NaN,NaN,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.000000,1
3,2016,2016-10-25,21600002,1610612762,1610612757,0,NaN,NaN,NaN,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.000000,0
4,2016,2016-10-25,21600003,1610612744,1610612759,1,NaN,NaN,NaN,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23953,2025,2026-04-12,22501198,1610612762,1610612747,0,-4.151206,5.517012,-9.668219,-0.361446,...,3.679012,3.172840,-0.024691,0.382716,2.370370,-0.728395,19.0,-0.404762,-0.265236,0
23954,2025,2026-04-12,22501199,1610612746,1610612744,1,2.465788,0.201384,2.264405,0.048193,...,-1.876543,-0.901235,-1.604938,-0.358025,0.444444,1.543210,-2.0,0.285714,0.048949,1
23955,2025,2026-04-12,22501199,1610612744,1610612746,0,-2.465788,-0.201384,-2.264405,-0.048193,...,1.876543,0.901235,1.604938,0.358025,-0.444444,-1.543210,2.0,-0.285714,-0.048949,0
23956,2025,2026-04-12,22501200,1610612757,1610612758,1,1.499363,-6.155883,7.655246,0.228916,...,-1.925926,-1.617284,1.580247,2.135802,0.962963,3.765432,-8.0,0.166667,0.255901,1


## Sanity checks

Same checks as before — `Games_Played` should start at 0 and increment by 1, and `Avg_Score` for any given pre-game row should equal the manual mean of that team's prior games' scores.

In [12]:
gp_check = (pregame.sort_values(['TeamID','Season','DayDate','GameID'])
                   .groupby(['TeamID','Season'])['Games_Played']
                   .agg(['min','max','count']))
print("Games_Played min should be 0 for every team-season:",
      (gp_check['min'] == 0).all())
print("Games_Played max should equal count - 1:",
      (gp_check['max'] == gp_check['count'] - 1).all())

team_log = (explode_to_team_perspective(box)
            .sort_values(['TeamID','Season','DayDate','GameID']))
sample = (pregame[pregame['Games_Played'] >= 4]
          .sort_values(['TeamID','Season','DayDate','GameID'])
          .head(1).iloc[0])

prior = team_log[(team_log['TeamID'] == sample['TeamID']) &
                 (team_log['Season'] == sample['Season']) &
                 (team_log['DayDate'] < sample['DayDate'])]
print(f"\nSpot check for TeamID={sample['TeamID']}, Season={sample['Season']}, "
      f"GameID={sample['GameID']}")
print(f"  Pre-game Avg_Score in pregame:        {sample['Avg_Score']:.3f}")
print(f"  Manual mean of prior games' Score:    {prior['Score'].mean():.3f}")

Games_Played min should be 0 for every team-season: True
Games_Played max should equal count - 1: True

Spot check for TeamID=1610612737, Season=2016, GameID=21600070
  Pre-game Avg_Score in pregame:        110.000
  Manual mean of prior games' Score:    110.000


## Done

You now have:

- `box_data.csv` — game-level T1/T2 box scores from the last ~10 NBA seasons
- `matchups.csv` — pre-game features for both teams, ready for modeling

`raw_seasons/` holds per-season caches in case you want to refresh just the current season later.

**To refresh the current season**: delete `raw_seasons/2025-26_Regular_Season.csv` and re-run the fetch loop. The other seasons stay cached.
